In [ ]:
from IPython import display
import pandas as pd

df = pd.read_csv("dataset/medquad.csv")

display(df.head())

,question,answer,source,focus_area
0,What is (are) Glaucoma ?,Glaucoma is a group of diseases that can damag...,NIHSeniorHealth,Glaucoma
1,What causes Glaucoma ?,"Nearly 2.7 million people have glaucoma, a lea...",NIHSeniorHealth,Glaucoma
2,What are the symptoms of Glaucoma ?,Symptoms of Glaucoma Glaucoma can develop in ...,NIHSeniorHealth,Glaucoma
3,What are the treatments for Glaucoma ?,"Although open-angle glaucoma cannot be cured, ...",NIHSeniorHealth,Glaucoma
4,What is (are) Glaucoma ?,Glaucoma is a group of diseases that can damag...,NIHSeniorHealth,Glaucoma


In [2]:
df.shape

(16412, 4)

In [3]:
import torch
from sentence_transformers import SentenceTransformer

if torch.backends.mps.is_available():
    device = "mps"
elif torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"
print(f"Using hardware accelerator: {device}")

MODEL_NAME = "NeuML/pubmedbert-base-embeddings"

Using hardware accelerator: mps


In [4]:
model = SentenceTransformer(MODEL_NAME, device=device)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/6.33k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/667 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/226k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/706k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/74.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [5]:
# Combine text columns natively
df["combined_text"] = df["question"].fillna("").astype(str) + " " + df["answer"].fillna("").astype(str)
texts = df["combined_text"].tolist()

print("Generating vectors...")
# Generate everything through the model's optimized internal engine
vectors = model.encode(texts, batch_size=64, show_progress_bar=True, normalize_embeddings=True)
df["embedding"] = list(vectors)

Generating vectors...


Batches:   0%|          | 0/257 [00:00<?, ?it/s]

In [7]:
df.to_csv("dataset/embeddings.csv", index=False)

In [8]:
df.head()

,question,answer,source,focus_area,combined_text,embedding
0,What is (are) Glaucoma ?,Glaucoma is a group of diseases that can damag...,NIHSeniorHealth,Glaucoma,What is (are) Glaucoma ? Glaucoma is a group o...,"[-0.014393196, 0.043612506, 0.015650649, -0.06..."
1,What causes Glaucoma ?,"Nearly 2.7 million people have glaucoma, a lea...",NIHSeniorHealth,Glaucoma,What causes Glaucoma ? Nearly 2.7 million peop...,"[-0.0022952422, 0.014643544, 0.0016189984, -0...."
2,What are the symptoms of Glaucoma ?,Symptoms of Glaucoma Glaucoma can develop in ...,NIHSeniorHealth,Glaucoma,What are the symptoms of Glaucoma ? Symptoms o...,"[-0.005584324, 0.022126397, 0.0056417244, -0.0..."
3,What are the treatments for Glaucoma ?,"Although open-angle glaucoma cannot be cured, ...",NIHSeniorHealth,Glaucoma,What are the treatments for Glaucoma ? Althoug...,"[-0.014800721, 0.030092023, 0.015314933, -0.07..."
4,What is (are) Glaucoma ?,Glaucoma is a group of diseases that can damag...,NIHSeniorHealth,Glaucoma,What is (are) Glaucoma ? Glaucoma is a group o...,"[-0.013924251, 0.030114612, 0.0051994193, -0.0..."
